# Russian Hotel Review Sentiment Analysis (2026)
## Reproduction of the 2021 methodology and a contemporary English-language research extension

**Reference study:** Elgeldawi, E., Sayed, A., Galal, A. R., & Zaki, A. M. (2021). *Hyperparameter Tuning for Machine Learning Algorithms Used for Arabic Sentiment Analysis*. **Informatics**, 8(4), 79. [DOI: 10.3390/informatics8040079](https://doi.org/10.3390/informatics8040079).

**Our study:** Russian-language hotel sentiment classification using a balanced corpus of **5,608** reviews (2,804 negative; 2,804 positive). The labels were **derived from Yandex user ratings**, not from manual sentiment annotation. Source: [Yandex Geo Reviews Dataset 2023](https://huggingface.co/datasets/d0rj/geo-reviews-dataset-2023), reviews dated January–July 2023. The 5,608-row filtered sample and a 76-word, sentiment-aware Russian stop-word list are stored in this public GitHub repository and downloaded at runtime; no manual CSV upload is necessary.

**Aims:**
1. Re-evaluate six original classifier families (LR, RC, DT, RF, SVC and GaussianNB) using Russian texts, stratified cross-validation and an untouched holdout set.
2. Compare Grid, Random, Bayesian, PSO and GA tuning with documented budgets.
3. Produce *analogues* of article Figures 5–19 and modern visualization Figures 20–32: bootstrap uncertainty, trade-offs, class-wise errors, calibration, convergence, significance tests, Russian lexical features, and stop-word ablation.
4. Export editable/vector graphics (PNG at 300 dpi, SVG and PDF), result CSVs and experiment metadata.

**Scientific caveats:** This is a **cross-language adaptation, not an exact numerical replication**. The original Arabic corpus used 7,000 manually labeled Booking.com reviews; ours uses 5,608 Russian reviews with rating-derived labels. The two corpora cannot be compared as a controlled test of language alone. The supplied CSV has `text,label` only, so hotel-level identifiers and exact ratings are **not available** for group-level leakage audits. We deduplicate identical normalized texts before splitting, but cannot rule out shared-hotel leakage. Scores reported during tuning are **training-set CV** results, distinct from the final holdout metrics. Original published figures are used only as a clearly labeled historical reference.

**Language of paper, code comments and charts: English.** Raw reviews and extracted lexical features remain in Russian.

**Colab:** Runtime → Run all. Set `PROFILE='quick'` for a preliminary check; switch to `PROFILE='full'` for a larger search budget and tuning of all six models (computationally expensive).


## 1 · Setup and experimental settings

The dataset and Russian stop words are retrieved from the public project GitHub repository. The dataset is delivered in numbered CSV parts to support reproducible transfer. Each estimator contains its own TF-IDF vectorizer, fitted **inside** the CV fold. Russian negation terms such as *не*, *нет*, *без*, *ни* and *но* must not be deleted. A `quick` configuration tunes LR, RC and SVC; `full` tunes all six models. No results from the previous Arabic experiment are imported.

In [ ]:
# Bayesian optimization is optional; install scikit-optimize in Colab if absent.
import sys, subprocess, importlib.util
if importlib.util.find_spec('skopt') is None:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-optimize'])
    except Exception as exc:
        print('Optional Bayesian package not installed; Bayesian search will be skipped:', exc)
HAS_SKOPT = importlib.util.find_spec('skopt') is not None

import os, re, time, json, hashlib, warnings, html
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter
from scipy import sparse
from scipy.stats import norm
from IPython.display import display
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, cross_val_predict, ParameterGrid,
                                     ParameterSampler, learning_curve)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             precision_score, recall_score, confusion_matrix,
                             roc_auc_score, brier_score_loss, log_loss,
                             classification_report)
from sklearn.calibration import calibration_curve
if HAS_SKOPT:
    from skopt import BayesSearchCV
    from skopt.space import Categorical

PROFILE = 'quick'         # 'quick' or 'full'; full substantially more expensive
RUN_SEARCH = True         # True: Grid / Random / Bayesian / PSO / GA
RUN_MODERN = True         # True: character n-grams + scientific figures
RUN_LEARNING_CURVE = False  # separate optional computation (not in quick profile)
DATA_PATH = None         # None: retrieve the repository CSV dataset; optionally override with your own text/label CSV
STOPWORDS_PATH = None    # None: retrieve the repository stop-word CSV
USE_STOPWORDS = True     # False: disable Russian stop-word removal in all word-level TF-IDF pipelines
RANDOM_SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 3 if PROFILE == 'quick' else 5
MAX_FEATURES = 2800 if PROFILE == 'quick' else 6500
SEARCH_BUDGET = 5 if PROFILE == 'quick' else 15
GA_POP = 3 if PROFILE == 'quick' else 5
GA_GENS = 2 if PROFILE == 'quick' else 3
PSO_POP = 3 if PROFILE == 'quick' else 5
PSO_GENS = 2 if PROFILE == 'quick' else 3
BOOTSTRAP_SAMPLES = 600 if PROFILE == 'quick' else 1500
# In quick mode tune fast models and SVC. Set to all six for complete paper-method comparison.
TUNE_MODELS = ['LR', 'RC', 'SVC'] if PROFILE == 'quick' else ['LR','RC','DT','RF','SVC','NB']
OUT = Path('RHSA_2026_outputs'); OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT / 'figures'; FIG_DIR.mkdir(exist_ok=True)
RNG = np.random.default_rng(RANDOM_SEED)

print('PROFILE:', PROFILE, '| CV folds:', CV_FOLDS, '| Tuning:', TUNE_MODELS)
print('Figures saved to:', FIG_DIR.resolve())
print('Corpus: Russian hotel reviews (Yandex Geo Reviews 2023); rating-derived binary labels.')


In [ ]:
# Download the published corpus and stop-word list from GitHub.
# The corpus is stored in numbered, independently parseable CSV shards.
import urllib.request
from pathlib import Path
REPOSITORY = 'Koyd1/RHSA_2026'
REVISION = 'main'
RAW_BASE = f'https://raw.githubusercontent.com/{REPOSITORY}/{REVISION}'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
CSV_SHARDS = 7
data_target = DATA_DIR / 'russian_hotel_ML_5608.csv'
stops_target = DATA_DIR / 'stopWords_ru_sentiment.csv'
if DATA_PATH is None and not data_target.exists():
    frames = []
    for i in range(1, CSV_SHARDS + 1):
        url = f'{RAW_BASE}/data/parts/reviews_{i:02d}.csv'
        print('Downloading dataset segment:', i, '/', CSV_SHARDS)
        frames.append(pd.read_csv(url, encoding='utf-8-sig'))
    restored = pd.concat(frames, ignore_index=True)
    if not {'text', 'label'}.issubset(restored.columns):
        raise RuntimeError('Required dataset fields missing')
    if len(restored) != 5608 or set(restored['label'].dropna().astype(int)) != {0,1}:
        raise RuntimeError('Unexpected dataset size or labels')
    restored.to_csv(data_target, index=False, encoding='utf-8-sig')
    print('Dataset restored:', len(restored), 'rows')
if STOPWORDS_PATH is None and not stops_target.exists():
    urllib.request.urlretrieve(f'{RAW_BASE}/data/stopWords_ru_sentiment.csv', stops_target)
print('Data directory:', DATA_DIR.resolve())


## 2 · Russian-language dataset, provenance and split

The source CSV is a **previously extracted subset of Yandex Geo Reviews 2023** restricted to accommodation-related organizations. It contains **5,608 rows** with only `text` and `label`: `0` is a 1–2-star review (negative proxy), `1` is a 4–5-star review (positive proxy); neutral and unrated records are absent. Exact rating and accommodation metadata are not included in the supplied ML CSV. These labels are **rating-derived proxies**, not verified manual sentiment labels; mixed-sentiment reviews may be mislabeled.

Russian normalization is **stateless** and deliberately conservative: HTML unescaping, lowercasing, whitespace and punctuation cleanup. It **preserves Cyrillic, `ё`, numbers, Latin tokens and all negation words**. No lemmatization is imposed. Exact duplicate and label-conflicting normalized texts are removed before splitting; the resulting retention count is reported. Stratified 80/20 holdout is allocated **before** any TF-IDF fitting or optimization.

In [ ]:
DATASET_SOURCE = 'https://huggingface.co/datasets/d0rj/geo-reviews-dataset-2023'
DATASET_METHODOLOGY = 'Hotel/accommodation subset extracted from Yandex Geo Reviews 2023; rating-derived labels (1–2 -> 0; 4–5 -> 1).'

# By default, the two input CSVs are downloaded from the GitHub repository in the previous cell.
def locate_data_file(override, candidates, label):
    if override is not None:
        p = Path(override)
        if not p.exists(): raise FileNotFoundError(f'{label} override does not exist: {p}')
        return p
    for p in candidates:
        if Path(p).exists(): return Path(p)
    raise FileNotFoundError(f'{label} is missing. Run the GitHub data-download cell first.')

source_path = locate_data_file(DATA_PATH, [
    'data/russian_hotel_ML_5608.csv', '/content/data/russian_hotel_ML_5608.csv',
    'russian_hotel_ML_5608.csv'
], 'Russian hotel reviews')
stopwords_path = locate_data_file(STOPWORDS_PATH, [
    'data/stopWords_ru_sentiment.csv', '/content/data/stopWords_ru_sentiment.csv',
    'stopWords_ru_sentiment.csv'
], 'Russian stop-words')

raw = pd.read_csv(source_path, encoding='utf-8-sig')
if not {'text','label'}.issubset(raw.columns):
    raise ValueError(f'Required fields text,label are missing: {list(raw.columns)}')
source_rows = len(raw)
raw['label'] = pd.to_numeric(raw['label'], errors='coerce')
raw = raw.dropna(subset=['text','label']).copy()
raw = raw[raw['label'].isin([0,1])].copy()
raw['label'] = raw['label'].astype(int)
raw['text'] = raw['text'].astype(str).str.strip()
raw = raw.loc[raw.text.str.len()>=2].copy()
pre_duplicate_count = len(raw)
raw = raw.drop_duplicates(subset=['text','label']).copy()
ambiguous = raw.groupby('text').label.nunique()
ambiguous = set(ambiguous.loc[ambiguous>1].index)
raw = raw[~raw.text.isin(ambiguous)].copy()

stops = pd.read_csv(stopwords_path, encoding='utf-8-sig')
if stops.empty: raise ValueError('The stop-word CSV is empty')
STOP_WORDS = sorted(set(stops.iloc[:,0].dropna().astype(str).str.strip().str.casefold()) - {''})
PROTECTED = {'не','нет','ни','без','но','только','однако'}
if PROTECTED.intersection(STOP_WORDS):
    raise ValueError(f'Sentiment-critical negation/contrast terms in stop words: {PROTECTED.intersection(STOP_WORDS)}')

# Stateless normalization; do not destroy informative Russian negation.
def normalize_russian(text):
    s = html.unescape(str(text)).casefold()
    s = re.sub(r'https?://\S+|www\.\S+', ' url ', s)
    s = re.sub(r'[^\w\s\-]', ' ', s, flags=re.UNICODE)
    s = re.sub(r'_', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

texts_norm = raw.text.map(normalize_russian)
keep = texts_norm.str.len()>=2
norm_df = pd.DataFrame({'text':texts_norm.loc[keep].to_numpy(),
                        'label':raw.loc[keep,'label'].to_numpy(dtype=int)})
conflicts = norm_df.groupby('text').label.nunique()
conflicts = set(conflicts.loc[conflicts>1].index)
norm_df = norm_df[~norm_df.text.isin(conflicts)].drop_duplicates('text').reset_index(drop=True)
norm_df['word_count'] = norm_df.text.str.split().map(len)
texts, labels, record_lengths = norm_df.text, norm_df.label, norm_df.word_count
if len(texts)<100 or labels.nunique()!=2:
    raise ValueError('The cleaned dataset has too few labeled examples of both classes')

print('SOURCE:', DATASET_SOURCE)
print('CSV rows:', source_rows, '| retained:',len(norm_df),'| duplicates removed:',pre_duplicate_count-len(raw))
print('Conflicting normalized texts removed:', len(conflicts))
print('Labels: 0 = Negative (rating 1–2), 1 = Positive (rating 4–5)')
print('Class counts:',labels.value_counts().sort_index().to_dict())
print('Russian stop words:',len(STOP_WORDS), '| enabled:',USE_STOPWORDS)
print('Stop-words example:',STOP_WORDS[:20])
if len(norm_df)!=source_rows:
    warnings.warn('Cleaning changed the number of rows; class counts and outputs must use the retained sample size.')
if any(not bool(re.search(r'[а-яё]',str(t))) for t in texts.sample(min(50,len(texts)),random_state=RANDOM_SEED)):
    warnings.warn('Some texts do not appear to contain Russian Cyrillic; inspect your input.')

display(norm_df[['text','label','word_count']].head(4))
X_train_txt,X_test_txt,y_train,y_test = train_test_split(
    texts,labels,test_size=TEST_SIZE,stratify=labels,random_state=RANDOM_SEED)
y_train=np.asarray(y_train,dtype=int);y_test=np.asarray(y_test,dtype=int)
print(f'Train: {len(y_train):,} | untouched test: {len(y_test):,}')


## 3 · Russian word- and character-level TF-IDF pipelines

The principal pipeline uses sentiment-aware Russian stop words for **word** n-grams (1–2); the character n-gram extension uses characters (3–5) and **does not** remove stop words. The original paper’s six classifier families are preserved. `GaussianNB` requires dense features: the conversion happens only after feature capping (and may be slower). `MultinomialNB` and `LinearSVC` appear only in a clearly labeled *modern extension*. All TF-IDF vocabularies and inverse document frequencies are fit **within** the CV training fold.

In [ ]:
class ToDense(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X.toarray() if sparse.issparse(X) else np.asarray(X)

BASE_MODELS = {
    'LR': LogisticRegression(max_iter=600, solver='lbfgs', random_state=RANDOM_SEED),
    'RC': RidgeClassifier(),
    'DT': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'RF': RandomForestClassifier(n_estimators=80, n_jobs=1, random_state=RANDOM_SEED),
    'SVC': SVC(kernel='rbf', gamma='scale', cache_size=300),
    'NB': GaussianNB(),
}
ORDER = ['RC','DT','RF','SVC','LR','NB']
METHOD_ORDER = ['Default','Grid','Random','Bayesian','PSO','GA']

SPACE = {
    'LR': {'model__C':[0.1,1.0,10.0,30.0]},
    'RC': {'model__alpha':[0.1,0.5,1,3,10]},
    'DT': {'model__max_depth':[8,15,25,40], 'model__min_samples_leaf':[1,3]},
    'RF': {'model__n_estimators':[40,80,120], 'model__max_depth':[15,30]},
    'SVC':{'model__C':[0.1,1,3,10], 'model__kernel':['linear','rbf']},
    'NB': {'model__var_smoothing':[1e-9,1e-7,1e-5,1e-4]},
}

def build_estimator(name, params=None, char=False):
    vector = TfidfVectorizer(
        analyzer='char' if char else 'word',
        ngram_range=(3,5) if char else (1,2),
        max_features=MAX_FEATURES,
        min_df=2,
        dtype=np.float32,
        sublinear_tf=True,
        stop_words=None if char or not USE_STOPWORDS else STOP_WORDS,
        token_pattern=r'(?u)\b\w\w+\b'
    )
    if name == 'CHAR-LSVC':
        model = LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_SEED)
    elif name == 'MNB':
        model = MultinomialNB(alpha=0.5)
    else:
        model = clone(BASE_MODELS[name])
    steps = [('tfidf', vector)]
    if name == 'NB': steps.append(('dense', ToDense()))
    steps.append(('model', model))
    est = Pipeline(steps)
    return est.set_params(**(params or {}))

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

## 4 · Baseline models: CV estimates and independent test predictions

The boxplot uses **training-fold CV** accuracy. The held-out test set is used to report Accuracy, balanced accuracy and Macro-F1 once per fitted baseline model. The six confusion matrices and calibration curves are from **actual held-out predictions**, never from published article values.

In [ ]:
BASE_FOLDS, BASE_TEST, BASE_PREDS, FITTED_BASE, TEST_PROBS = [], [], {}, {}, {}
for name in ORDER:
    start=time.perf_counter()
    print(f'[baseline] {name}',flush=True)
    est=build_estimator(name)
    result=cross_validate(est,X_train_txt,y_train,scoring={'acc':'accuracy','f1':'f1_macro'},cv=cv,n_jobs=1)
    BASE_FOLDS += [{'Model':name,'Fold':i+1,'Accuracy':float(a),'Macro-F1':float(f)}
                   for i,(a,f) in enumerate(zip(result['test_acc'],result['test_f1']))]
    fitted=est.fit(X_train_txt,y_train)
    pred=fitted.predict(X_test_txt)
    BASE_PREDS[name]=pred; FITTED_BASE[name]=fitted
    BASE_TEST.append({'Model':name,'Method':'Default','Accuracy':accuracy_score(y_test,pred),
                      'Macro-F1':f1_score(y_test,pred,average='macro'),
                      'Balanced Accuracy':balanced_accuracy_score(y_test,pred),
                      'Time (s)':time.perf_counter()-start})
    if hasattr(fitted, 'predict_proba'):
        try: TEST_PROBS[name] = fitted.predict_proba(X_test_txt)[:,1]
        except Exception: pass
    print(f'  CV: {np.mean(result["test_acc"]):.3f} | test: {BASE_TEST[-1]["Accuracy"]:.3f} | {time.perf_counter()-start:.1f}s',flush=True)
base_folds=pd.DataFrame(BASE_FOLDS)
base_test=pd.DataFrame(BASE_TEST)
base_folds.to_csv(OUT/'baseline_cv_folds.csv',index=False)
base_test.to_csv(OUT/'baseline_test.csv',index=False)
display(base_test.sort_values('Accuracy',ascending=False).round(3))

## 5 · Hyperparameter search: Grid, Random, Bayesian, PSO and GA

Hyperparameters are selected solely by **training-fold CV accuracy**. The five methods are run with modest but visible computational budgets; these are *budget-matched approximations*, **not** the exact 2021 optimization settings. `quick` tunes LR, RC and SVC; `full` includes all six families. The holdout set remains untouched until the selected best model is evaluated. Any unavailable searches appear as `N/A`, never as fabricated values.

In [ ]:
SEARCH_HISTORY=[]; SEARCH_SUMMARY=[]; BEST_CV={}; BEST_FITTED={}; BEST_PRED={}; SEARCH_TEST=[]

def record(name, method, step, cfg, cv_mean, cv_std, elapsed, from_cache=False):
    row={'Model':name,'Method':method,'Eval':int(step),'CV Accuracy':float(cv_mean),
         'CV SD':float(cv_std),'Elapsed (s)':float(elapsed),'Parameters':json.dumps(cfg,ensure_ascii=False,default=str),
         'Cached':bool(from_cache)}
    SEARCH_HISTORY.append(row)

def evaluation_fn(name, method):
    cache={}; start=time.perf_counter(); call=[0]
    def score(config):
        call[0]+=1; key=json.dumps(config,sort_keys=True,default=str)
        cached=key in cache
        if not cached:
            e=build_estimator(name,config)
            r=cross_validate(e,X_train_txt,y_train,cv=cv,scoring='accuracy',n_jobs=1,error_score='raise')
            cache[key]=(float(r['test_score'].mean()),float(r['test_score'].std()))
        m,s=cache[key]
        record(name,method,call[0],config,m,s,time.perf_counter()-start,cached)
        print(f'  {name}/{method} [{call[0]}]: CV Accuracy={m*100:.2f}%, best-so-far={max(v[0] for v in cache.values())*100:.2f}%',flush=True)
        return m
    return score,cache,start

def choose_from_position(values, position):
    # Normalized chromosome [0,1], deterministic discrete encoding.
    return {k:vals[min(len(vals)-1,int(float(np.clip(position[i],0,1))*len(vals)))]
            for i,(k,vals) in enumerate(values.items())}

def run_custom(name,method):
    dim=len(SPACE[name]); values=SPACE[name]
    score,cache,start=evaluation_fn(name,method)
    gen=np.random.default_rng(RANDOM_SEED + ORDER.index(name)+(0 if method=='PSO' else 500))
    if method=='PSO':
        pop=gen.random((PSO_POP,dim)); vel=np.zeros_like(pop)
        pbest=pop.copy(); pscore=np.full(PSO_POP,-np.inf)
        gbest=pop[0].copy(); gscore=-np.inf
        for it in range(PSO_GENS):
            for i in range(PSO_POP):
                val=score(choose_from_position(values,pop[i]))
                if val>pscore[i]: pscore[i],pbest[i]=val,pop[i].copy()
                if val>gscore: gscore,gbest=val,pop[i].copy()
            vel=.55*vel + 1.4*gen.random(pop.shape)*(pbest-pop)+1.4*gen.random(pop.shape)*(gbest-pop)
            pop=np.clip(pop+vel,0,1)
    else:
        pop=gen.random((GA_POP,dim))
        for it in range(GA_GENS):
            scores=np.array([score(choose_from_position(values,p)) for p in pop])
            if it+1==GA_GENS: break
            elite=pop[int(scores.argmax())].copy()
            children=[elite]
            while len(children)<GA_POP:
                ix=gen.integers(0,GA_POP,4)
                p1=pop[ix[:2][np.argmax(scores[ix[:2]])]]
                p2=pop[ix[2:][np.argmax(scores[ix[2:]])]]
                child=np.where(gen.random(dim)<.5,p1,p2)
                if gen.random()<.25:
                    child[int(gen.integers(dim))]=gen.random()
                children.append(child)
            pop=np.asarray(children)
    best_key=max(cache,key=lambda k:cache[k][0]); best_config=json.loads(best_key)
    return best_config,cache[best_key][0],time.perf_counter()-start

if RUN_SEARCH:
    for name in TUNE_MODELS:
        space=SPACE[name]
        for method in ['Grid','Random','Bayesian','PSO','GA']:
            print(f'\n=== {name} / {method} ===',flush=True)
            started=time.perf_counter()
            try:
                if method in ['Grid','Random']:
                    candidates=list(ParameterGrid(space)) if method=='Grid' else list(ParameterSampler(space,n_iter=min(SEARCH_BUDGET,len(list(ParameterGrid(space)))),random_state=RANDOM_SEED))
                    if method=='Grid' and PROFILE=='quick' and len(candidates)>SEARCH_BUDGET:
                        # Approximate grid in quick profile, explicitly labeled as truncated.
                        ix=np.unique(np.round(np.linspace(0,len(candidates)-1,SEARCH_BUDGET)).astype(int))
                        candidates=[candidates[i] for i in ix]
                    score,cache,_=evaluation_fn(name,method)
                    for c in candidates: score(c)
                    best_key=max(cache,key=lambda k:cache[k][0]); params=json.loads(best_key)
                    best_score=cache[best_key][0]
                elif method=='Bayesian':
                    if not HAS_SKOPT: raise RuntimeError('scikit-optimize unavailable')
                    bayes_space={k:Categorical(v) for k,v in space.items()}
                    search=BayesSearchCV(build_estimator(name),bayes_space,n_iter=SEARCH_BUDGET,
                        cv=cv,scoring='accuracy',n_jobs=1,random_state=RANDOM_SEED,refit=False,
                        optimizer_kwargs={'n_initial_points':min(4,SEARCH_BUDGET)})
                    search.fit(X_train_txt,y_train)
                    for i,(param,mean,std) in enumerate(zip(search.cv_results_['params'],search.cv_results_['mean_test_score'],search.cv_results_['std_test_score'])):
                        record(name,method,i+1,dict(param),mean,std,time.perf_counter()-started)
                    params=dict(search.best_params_);best_score=float(search.best_score_)
                else:
                    params,best_score,_=run_custom(name,method)
                elapsed=time.perf_counter()-started
                SEARCH_SUMMARY.append({'Model':name,'Method':method,'CV Accuracy':best_score,'Time (s)':elapsed,
                      'Parameters':json.dumps(params,ensure_ascii=False,default=str), 'Status':'ok'})
                if name not in BEST_CV or best_score>BEST_CV[name]['CV Accuracy']:
                    BEST_CV[name]={'CV Accuracy':best_score,'Method':method,'Parameters':params}
                print(f'BEST {name} {method}: {best_score:.4f} | {elapsed:.1f}s',flush=True)
            except Exception as e:
                print(f'SKIPPED {name}/{method}: {type(e).__name__} {str(e)[:200]}',flush=True)
                SEARCH_SUMMARY.append({'Model':name,'Method':method,'CV Accuracy':np.nan,'Time (s)':time.perf_counter()-started,
                       'Parameters':'','Status':f'{type(e).__name__}: {e}'})
            pd.DataFrame(SEARCH_SUMMARY).to_csv(OUT/'search_summary.csv',index=False)
            pd.DataFrame(SEARCH_HISTORY).to_csv(OUT/'search_history.csv',index=False)

for name, best in BEST_CV.items():
    model=build_estimator(name,best['Parameters']).fit(X_train_txt,y_train)
    predictions=model.predict(X_test_txt)
    BEST_FITTED[name]=model;BEST_PRED[name]=predictions
    SEARCH_TEST.append({'Model':name,'Method':best['Method'],'Accuracy':accuracy_score(y_test,predictions),
            'Macro-F1':f1_score(y_test,predictions,average='macro'),
            'Balanced Accuracy':balanced_accuracy_score(y_test,predictions),
            'CV Accuracy':best['CV Accuracy']})
search_summary=pd.DataFrame(SEARCH_SUMMARY,columns=['Model','Method','CV Accuracy','Time (s)','Parameters','Status'])
search_history=pd.DataFrame(SEARCH_HISTORY,columns=['Model','Method','Eval','CV Accuracy','CV SD','Elapsed (s)','Parameters','Cached'])
tuned_test=pd.DataFrame(SEARCH_TEST,columns=['Model','Method','Accuracy','Macro-F1','Balanced Accuracy','CV Accuracy'])
tuned_test.to_csv(OUT/'best_tuned_holdout.csv',index=False)
print('\nSearch summaries (CV); holdout evaluation once per winning model:')
display(search_summary.sort_values('CV Accuracy',ascending=False).head(15))
display(tuned_test)

## 6 · Publication-oriented visual styling

The plotting template uses English figure labels, subdued colors, vector-safe text and a consistent statistical vocabulary. Original Russian-language reviews and the model’s learned Cyrillic features are retained. Charts show measured outcomes rather than invented values. Export: 300-dpi PNG + vector SVG/PDF.

In [ ]:
INK='#253341'; TEAL='#087e83'; GOLD='#cf934b'; CORAL='#b95756'; PURPLE='#70669a'; GREY='#88939d'; LIGHT='#e7ebed'
PALETTE={'Default':GREY,'Grid':PURPLE,'Random':GOLD,'Bayesian':TEAL,'PSO':CORAL,'GA':'#497b59'}
MODEL_COLORS={'RC':TEAL,'DT':GOLD,'RF':PURPLE,'SVC':'#365c94','LR':CORAL,'NB':'#79975b'}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9.5,'axes.titlesize':11,'axes.labelsize':9,
 'axes.edgecolor':'#aeb7bc','axes.linewidth':.75,'axes.spines.top':False,'axes.spines.right':False,
 'xtick.color':INK,'ytick.color':INK,'axes.labelcolor':INK,'text.color':INK,'figure.facecolor':'white',
 'axes.facecolor':'white','savefig.facecolor':'white','svg.fonttype':'none','pdf.fonttype':42,
 'grid.color':'#e5e9eb','grid.linewidth':.7,'lines.linewidth':1.7})

def export(fig, filename):
    fig.tight_layout(pad=1.2)
    for ext in ['png','svg','pdf']:
        fig.savefig(FIG_DIR/f'{filename}.{ext}',dpi=300,bbox_inches='tight')
    plt.show();plt.close(fig)

def label_axes(ax, title, xlabel=None, ylabel=None):
    ax.set_title(title,loc='left',fontweight='bold',pad=12)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(axis='y',alpha=.8); ax.set_axisbelow(True)

## 7 · Figures 5–19: structural analogues of the original article

Figures 5–8: baseline performance, calibration and confusion matrices. Figures 9–19: original classifier families and tuning comparisons (missing measurements are explicitly marked). These are **visual analogues based on a different Russian corpus**, not copies of the Arabic study. Parameter-response figures depict observed search candidates, not controlled marginal effects.

In [ ]:
# Figure 5: baseline CV box plot, 6 classifiers.
fig,ax=plt.subplots(figsize=(7.6,4.1))
vals=[base_folds.loc[base_folds.Model==m,'Accuracy'].values*100 for m in ORDER]
bp=ax.boxplot(vals,positions=range(len(ORDER)),widths=.46,patch_artist=True,showmeans=True,
 meanprops={'marker':'D','markerfacecolor':INK,'markeredgecolor':INK,'markersize':4},
 medianprops={'color':INK,'linewidth':1.5})
for p,m in zip(bp['boxes'],ORDER):p.set(facecolor=MODEL_COLORS[m],alpha=.7,edgecolor='white')
ax.set_xticks(range(len(ORDER)),ORDER)
label_axes(ax,'FIGURE 05  |  Baseline CV accuracy',ylabel='Accuracy (%)')
export(fig,'Figure_05_Baseline_Boxplot')

# Figure 6: baseline test metrics. Accuracy and Macro-F1 are clearly distinct.
fig,ax=plt.subplots(figsize=(7.6,4.2))
pos=np.arange(len(ORDER));d=base_test.set_index('Model').reindex(ORDER)
ax.bar(pos-.17,d.Accuracy*100,.34,color=TEAL,label='Holdout accuracy')
ax.bar(pos+.17,d['Macro-F1']*100,.34,color=GOLD,label='Holdout macro-F1')
ax.set_xticks(pos,ORDER);ax.set_ylim(0,105); ax.legend(frameon=False,ncol=2)
label_axes(ax,'FIGURE 06  |  Default classifiers, held-out test',ylabel='Performance (%)')
export(fig,'Figure_06_Baseline_Accuracy')

# Figure 7: calibration curves, only classifiers exposing well-defined probabilities.
from sklearn.calibration import calibration_curve
fig,(ax,hist)=plt.subplots(2,1,figsize=(7.6,6),sharex=True,gridspec_kw={'height_ratios':[2.2,1]})
ax.plot([0,1],[0,1],ls='--',color=GREY,label='Perfect')
nbins=8
for m in ['LR','RF','NB']:
    if m in TEST_PROBS:
        p=TEST_PROBS[m]
        frac,mean=calibration_curve(y_test,p,n_bins=nbins,strategy='quantile')
        ax.plot(mean,frac,'o-',label=m,color=MODEL_COLORS[m],markersize=4)
        hist.hist(p,bins=np.linspace(0,1,11),histtype='step',lw=1.5,color=MODEL_COLORS[m],label=m)
ax.legend(frameon=False,ncol=4);ax.set_ylabel('Observed positive fraction');hist.set_ylabel('Predictions')
hist.set_xlabel('Predicted probability of positive');hist.set_xlim(0,1)
label_axes(ax,'FIGURE 07  |  Probability calibration (holdout)')
export(fig,'Figure_07_Calibration')

# Figure 8: confusion matrices, six original models, normalized by actual class.
fig,axs=plt.subplots(2,3,figsize=(10,6.8))
cm_cmap=LinearSegmentedColormap.from_list('cm',['#f1f5f5',TEAL])
for ax,m in zip(axs.flat,ORDER):
    mat=confusion_matrix(y_test,BASE_PREDS[m],labels=[0,1],normalize='true')*100
    ax.imshow(mat,cmap=cm_cmap,vmin=0,vmax=100)
    for i in range(2):
        for j in range(2):ax.text(j,i,f'{mat[i,j]:.1f}%',ha='center',va='center',color='white' if mat[i,j]>65 else INK,fontsize=11,weight='bold')
    ax.set_xticks([0,1],['Neg','Pos']);ax.set_yticks([0,1],['Neg','Pos'])
    ax.set_title(m,weight='bold');ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.grid(False)
fig.suptitle('FIGURE 08  |  Holdout confusion matrices, row-normalized',x=.03,ha='left',weight='bold')
export(fig,'Figure_08_Confusion_Matrices')

# Figures 9,11,13,15,17,19: tuned methods vs default CV accuracy, not holdout.
article_numbers={'RC':9,'DT':11,'RF':13,'SVC':15,'LR':17,'NB':19}
base_cv=base_folds.groupby('Model').Accuracy.mean()
for m,num in article_numbers.items():
    method_vals={'Default':float(base_cv[m])}
    sub=search_summary[(search_summary.Model==m)&(search_summary.Status=='ok')]
    for _,r in sub.iterrows(): method_vals[r.Method]=r['CV Accuracy']
    fig,ax=plt.subplots(figsize=(8,4.3)); xx=np.arange(len(METHOD_ORDER))
    for j,meth in enumerate(METHOD_ORDER):
        v=method_vals.get(meth,np.nan)
        if np.isfinite(v):
            ax.bar(j,v*100,color=PALETTE[meth],width=.65)
            ax.text(j,v*100+0.4,f'{v*100:.1f}',ha='center',fontsize=8)
        else:
            ax.text(j,2,'N/A',ha='center',fontsize=9,color=GREY)
    ax.set_xticks(xx,METHOD_ORDER);ax.set_ylim(0,105)
    label_axes(ax,f'FIGURE {num:02d}  |  {m} tuning comparison (train CV)',ylabel='Mean CV accuracy (%)')
    export(fig,f'Figure_{num:02d}_Tuning_{m}')

# Figures 10,12,14,16,18: response to first varied hyperparameter, grounded in actual evaluations.
parameter_numbers={'RC':10,'DT':12,'RF':14,'SVC':16,'LR':18}
for m,num in parameter_numbers.items():
    fig,axs=plt.subplots(1,2,figsize=(9.5,3.9))
    data=search_history[search_history.Model==m].copy()
    for k,ax in enumerate(axs):
        params=list(SPACE[m]);param=params[min(k,len(params)-1)]
        if data.empty:
            ax.text(.5,.5,'Not measured in quick profile',ha='center',va='center',transform=ax.transAxes,color=GREY)
        else:
            x=data.Parameters.map(lambda s:json.loads(s).get(param,np.nan) if isinstance(s,str) and s else np.nan)
            valid=pd.DataFrame({'val':x.astype(str),'score':data['CV Accuracy']}).dropna()
            grp=valid.groupby('val')['score'].agg(['mean','min','max','count']).reset_index()
            order=[str(v) for v in SPACE[m][param]]
            grp['val']=pd.Categorical(grp['val'],categories=order,ordered=True)
            grp=grp.sort_values('val').dropna(subset=['val'])
            if len(grp):
                pos=np.arange(len(grp))
                ax.errorbar(pos,grp['mean']*100,yerr=[(grp['mean']-grp['min'])*100,(grp['max']-grp['mean'])*100],
                            marker='o',color=TEAL,capsize=3)
                ax.set_xticks(pos,[str(v) for v in grp['val']])
            else:ax.text(.5,.5,'No matching evaluations',ha='center',va='center',transform=ax.transAxes)
        ax.set_xlabel(param.replace('model__',''));ax.set_ylabel('Mean CV accuracy (%)')
        ax.grid(axis='y',alpha=.7)
    fig.suptitle(f'FIGURE {num:02d}  |  {m} hyperparameter response (observed candidates)',ha='left',x=.02,fontweight='bold')
    export(fig,f'Figure_{num:02d}_Parameter_Response_{m}')

## 8 · Modern evidence visualizations (Figures 20–30)

In addition to the original structure, the notebook provides a data profile, bootstrap 95% intervals for held-out accuracy, accuracy/runtime trade-off, gains from tuning, class-conditional Precision/Recall/F1, error rate by review length, optimization convergence, reliability diagrams, Russian character n-grams and paired significance testing.

**Figure 30** displays our Russian training-CV scores next to the original Arabic results purely for historical context; its percentage-point differences **cannot be interpreted as a causal language effect**. Figures 31–32 below are Russian-specific additions.

In [ ]:
def boot_ci(y, predictions, metric=accuracy_score, n=BOOTSTRAP_SAMPLES):
    y=np.asarray(y); p=np.asarray(predictions); idx=np.arange(len(y))
    rng=np.random.default_rng(RANDOM_SEED+1001)
    values=np.empty(n)
    for i in range(n):
        j=rng.choice(idx,size=len(idx),replace=True)
        values[i]=metric(y[j],p[j])
    return np.quantile(values,[.025,.975])

def boot_delta(y, a, b, n=BOOTSTRAP_SAMPLES):
    y=np.asarray(y); a=np.asarray(a); b=np.asarray(b); ix=np.arange(len(y))
    rng=np.random.default_rng(RANDOM_SEED+2002); ds=[]
    for i in range(n):
        j=rng.choice(ix,size=len(ix),replace=True)
        ds.append(accuracy_score(y[j],b[j])-accuracy_score(y[j],a[j]))
    return np.quantile(ds,[.025,.975])

# 20 | balanced dataset and review length profile (not just label pie).
fig,axs=plt.subplots(1,2,figsize=(9.2,3.6),gridspec_kw={'width_ratios':[1,1.6]})
counts=pd.Series(labels).value_counts().reindex([0,1],fill_value=0)
axs[0].barh(['Negative','Positive'],counts.values,color=[CORAL,TEAL],height=.52)
for i,v in enumerate(counts.values):axs[0].text(v+counts.max()*.02,i,f'{v:,}',va='center')
axs[0].set_xlim(0,counts.max()*1.24)
label_axes(axs[0],'A  |  Labeled reviews',xlabel='Documents')
for lab,col in [(0,CORAL),(1,TEAL)]:
    w=np.asarray(record_lengths)[np.asarray(labels)==lab]
    axs[1].hist(w,bins=np.arange(0,min(np.percentile(record_lengths,99.5)+8,125),4),density=True,
                histtype='step',color=col,linewidth=2,label='Positive' if lab else 'Negative')
axs[1].legend(frameon=False)
label_axes(axs[1],'B  |  Text length by sentiment',xlabel='Words per review',ylabel='Density')
export(fig,'Figure_20_Corpus_Profile')

# 21 | forest plot holdout accuracy + bootstrap CI.
fig,ax=plt.subplots(figsize=(8,4.3));vals=[]
for m in ORDER:
    v=accuracy_score(y_test,BASE_PREDS[m]);lo,hi=boot_ci(y_test,BASE_PREDS[m]);vals.append((m,v,lo,hi))
for i,(m,v,lo,hi) in enumerate(vals):
    ax.plot([lo*100,hi*100],[i,i],color=MODEL_COLORS[m],lw=2.3)
    ax.plot(v*100,i,marker='o',color=MODEL_COLORS[m],ms=7)
    ax.text(hi*100-.15 if hi>.95 else hi*100+.15,i,f'{v*100:.1f}%',va='center',ha='right' if hi>.95 else 'left',fontsize=8)
ax.set_yticks(range(len(vals)),[r[0] for r in vals]);ax.invert_yaxis()
ax.set_xlim(max(0,min(v[2] for v in vals)*100-5),min(100.5,max(v[3] for v in vals)*100+5))
label_axes(ax,'FIGURE 21  |  Accuracy and bootstrap uncertainty',xlabel='Holdout accuracy (%)')
export(fig,'Figure_21_Accuracy_Bootstrap_Forest')

# 22 | empirical time vs CV-accuracy with Pareto highlights; only measured searches.
fig,ax=plt.subplots(figsize=(8,4.8))
valid=search_summary.dropna(subset=['CV Accuracy','Time (s)']).copy()
if len(valid):
    valid['Pareto']=False
    for i,r in valid.iterrows():
        valid.at[i,'Pareto']=not ((valid['Time (s)'] < r['Time (s)']) & (valid['CV Accuracy']>=r['CV Accuracy']) |
                                   (valid['Time (s)'] <= r['Time (s)']) & (valid['CV Accuracy']>r['CV Accuracy'])).any()
    for method,grp in valid.groupby('Method'):
        ax.scatter(grp['Time (s)'],grp['CV Accuracy']*100,label=method,color=PALETTE.get(method,GREY),s=45,alpha=.85)
    top=valid[valid.Pareto]
    ax.scatter(top['Time (s)'],top['CV Accuracy']*100,facecolors='none',edgecolors=INK,s=130,lw=1.2,label='Pareto frontier')
    if (valid['Time (s)']>0).all():ax.set_xscale('log')
    ax.legend(frameon=False,fontsize=8,ncol=3,loc='best')
    ax.set_ylim(max(0,valid['CV Accuracy'].min()*100-4),min(100.5,valid['CV Accuracy'].max()*100+2))
else:
    ax.text(.5,.5,'Search not run',ha='center',va='center',transform=ax.transAxes)
label_axes(ax,'FIGURE 22  |  Quality–compute trade-off (search CV)',xlabel='Search wall-time (s; log scale)',ylabel='Mean CV accuracy (%)')
export(fig,'Figure_22_Pareto_Accuracy_Runtime')

# 23 | paired delta, with error bars.
fig,ax=plt.subplots(figsize=(8.6,4.0))
active=[m for m in ORDER if m in BEST_PRED]
for i,m in enumerate(active):
    start=accuracy_score(y_test,BASE_PREDS[m]); finish=accuracy_score(y_test,BEST_PRED[m])
    lo,hi=boot_delta(y_test,BASE_PREDS[m],BEST_PRED[m])
    ax.plot([start*100,finish*100],[i,i],color=GREY,lw=2,zorder=1)
    ax.scatter(start*100,i,color=GREY,s=55,zorder=2)
    ax.scatter(finish*100,i,color=TEAL,s=65,zorder=3)
    ax.annotate(f'Δ {(finish-start)*100:+.2f} pp  [{lo*100:+.2f}, {hi*100:+.2f}]',
                (max(start,finish)*100,i),xytext=(8,0),textcoords='offset points',
                annotation_clip=False,va='center',fontsize=7.6)
if active:
    ax.set_yticks(range(len(active)),active);ax.invert_yaxis()
    vv=[accuracy_score(y_test,p) for p in list(BASE_PREDS.values())+list(BEST_PRED.values())]
    ax.set_xlim(max(0,min(vv)*100-6),min(100.5,max(vv)*100+5))
    ax.scatter([],[],color=GREY,label='Default');ax.scatter([],[],color=TEAL,label='Tuned');ax.legend(frameon=False,ncol=2,loc='lower left')
else: ax.text(.5,.5,'No tuned models in this profile',ha='center',va='center',transform=ax.transAxes)
label_axes(ax,'FIGURE 23  |  Paired accuracy uplift (holdout)',xlabel='Holdout accuracy (%); Δ in percentage points')
export(fig,'Figure_23_Uplift_Dumbbell')

# 24 | class-based precision recall f1 heatmap for selected best models.
selected={m:BEST_PRED.get(m,BASE_PREDS[m]) for m in ORDER}
metrics=[]
for m,p in selected.items():
    for cls in [0,1]:
        metrics.append([m,cls,precision_score(y_test,p,labels=[cls],average='macro',zero_division=0),
                        recall_score(y_test,p,labels=[cls],average='macro',zero_division=0),
                        f1_score(y_test,p,labels=[cls],average='macro',zero_division=0)])
mat=np.array([r[2:] for r in metrics])*100
fig,ax=plt.subplots(figsize=(6.8,6.3));heat=ax.imshow(mat,cmap=LinearSegmentedColormap.from_list('performance',['#e9f0ee',TEAL]),vmin=50,vmax=100,aspect='auto')
for i in range(len(mat)):
    for j in range(3):ax.text(j,i,f'{mat[i,j]:.1f}',ha='center',va='center',color='white' if mat[i,j]>82 else INK,fontsize=8)
ax.set_xticks(range(3),['Precision','Recall','F1']);ax.set_yticks(range(len(metrics)),[f'{m} · {"positive" if cls else "negative"}' for m,cls,*_ in metrics]);ax.grid(False)
ax.set_title('FIGURE 24  |  Class-resolved holdout metrics',loc='left',fontweight='bold',pad=12)
fig.colorbar(heat,ax=ax,label='Score (%)',fraction=.035,pad=.03)
export(fig,'Figure_24_Classwise_Metric_Heatmap')

# 25 | length-bin Wilson intervals based on best overall by train CV (not holdout selection).
select_name=max(BEST_CV,key=lambda m:BEST_CV[m]['CV Accuracy']) if BEST_CV else base_folds.groupby('Model').Accuracy.mean().idxmax()
pred=BEST_PRED.get(select_name,BASE_PREDS[select_name]); lens=X_test_txt.str.split().map(len).to_numpy()
# Quantile bins, boundaries deduplicated automatically.
bins=pd.qcut(lens,q=6,duplicates='drop');rows=[]
for b in bins.categories:
    ids=np.asarray(bins==b);n=int(ids.sum())
    if n==0: continue
    k=int((pred[ids]!=y_test[ids]).sum());rate=k/n
    z=1.96;den=1+z*z/n;center=(rate+z*z/(2*n))/den;half=z*np.sqrt(rate*(1-rate)/n+z*z/(4*n*n))/den
    rows.append({'length':str(b),'error':rate*100,'lo':max(0,center-half)*100,'hi':min(1,center+half)*100,'n':n})
err_df=pd.DataFrame(rows);pos=np.arange(len(err_df))
fig,ax=plt.subplots(figsize=(8.5,4.4))
ax.errorbar(pos,err_df.error,yerr=[err_df.error-err_df.lo,err_df.hi-err_df.error],fmt='o-',color=CORAL,capsize=3)
for i,r in err_df.iterrows():ax.text(i,r['hi']+1.0,f'n={int(r["n"])}',ha='center',fontsize=8)
ax.set_xticks(pos,err_df.length,rotation=20,ha='right')
label_axes(ax,f'FIGURE 25  |  Errors vs length — {select_name}',xlabel='Review length (words, quantile bins)',ylabel='Holdout error rate (%) · Wilson 95% CI')
export(fig,'Figure_25_Error_By_Length')

# 26 | convergence CV as best-so-far, no made-up baseline interpolation.
fig,ax=plt.subplots(figsize=(8.6,4.5))
if len(search_history):
    view=search_history[search_history.Model==select_name]
    for meth,grp in view.groupby('Method'):
        s=grp.sort_values('Eval')
        ax.step(s['Eval'],s['CV Accuracy'].cummax()*100,where='post',label=meth,color=PALETTE.get(meth,GREY))
    ax.legend(frameon=False,ncol=3)
else:ax.text(.5,.5,'Search not run',ha='center',va='center',transform=ax.transAxes)
label_axes(ax,f'FIGURE 26  |  Optimization convergence — {select_name}',xlabel='Evaluations (budget per method)',ylabel='Best observed CV accuracy (%)')
export(fig,'Figure_26_Tuning_Convergence')

# 27 | reliability diagram, with Brier and ECE (post-hoc descriptive for noncalibrated probs).
fig,ax=plt.subplots(figsize=(7.8,4.6));ax.plot([0,1],[0,1],'--',color=GREY,label='Ideal')
for m in ['LR','RF','NB']:
    if m not in TEST_PROBS:continue
    p=TEST_PROBS[m];fq,mp=calibration_curve(y_test,p,n_bins=10,strategy='quantile')
    ids=np.minimum(np.floor(np.clip(p,0,1-1e-12)*10).astype(int),9)
    ece=sum((ids==k).mean()*abs(y_test[ids==k].mean()-p[ids==k].mean()) for k in range(10) if np.any(ids==k))
    ax.plot(mp,fq,'o-',color=MODEL_COLORS[m],markersize=4,label=f'{m} · Brier {brier_score_loss(y_test,p):.3f}, ECE {ece:.3f}')
ax.set_xlim(0,1);ax.set_ylim(0,1);ax.legend(frameon=False,fontsize=8)
label_axes(ax,'FIGURE 27  |  Probability reliability on holdout',xlabel='Predicted positive probability',ylabel='Observed positive fraction')
export(fig,'Figure_27_Probability_Reliability')

# 28 | modern character ngram experiment, extra models not silently substituted into originals.
modern_test=[]
if RUN_MODERN:
    for name,character in [('CHAR-LSVC',True),('MNB',False)]:
        begin=time.perf_counter();e=build_estimator(name,char=character).fit(X_train_txt,y_train)
        pp=e.predict(X_test_txt)
        modern_test.append({'Model':name,'Accuracy':accuracy_score(y_test,pp),
            'Macro-F1':f1_score(y_test,pp,average='macro'),'Time (s)':time.perf_counter()-begin})
modern_df=pd.DataFrame(modern_test,columns=['Model','Accuracy','Macro-F1','Time (s)'])
modern_df.to_csv(OUT/'modern_experiments.csv',index=False)
fig,ax=plt.subplots(figsize=(8.2,4.4))
all_bars=base_test[['Model','Accuracy','Macro-F1']].copy();all_bars=pd.concat([all_bars,modern_df[['Model','Accuracy','Macro-F1']]],ignore_index=True)
all_bars=all_bars.sort_values('Macro-F1',ascending=True)
ypos=np.arange(len(all_bars))
ax.barh(ypos-.18,all_bars.Accuracy*100,height=.34,color=TEAL,label='Accuracy')
ax.barh(ypos+.18,all_bars['Macro-F1']*100,height=.34,color=GOLD,label='Macro-F1')
ax.set_yticks(ypos,all_bars.Model);ax.set_xlim(0,110);ax.legend(frameon=False,ncol=2)
label_axes(ax,'FIGURE 28  |  Modern n-grams vs original classical models',xlabel='Held-out test performance (%)')
export(fig,'Figure_28_Modern_Char_Ngrams')
display(modern_df)

# 29 | exact paired McNemar comparisons among the original baseline classifiers.
# Multiple comparison correction uses Holm (family = all 15 distinct pairs).
from scipy.stats import binomtest
pair_p = []
for i in range(len(ORDER)):
    for j in range(i):
        a,b = BASE_PREDS[ORDER[i]],BASE_PREDS[ORDER[j]]
        a_correct=(a==y_test); b_correct=(b==y_test)
        n10=int(np.sum(a_correct & ~b_correct))
        n01=int(np.sum(~a_correct & b_correct))
        pval=float(binomtest(min(n10,n01),n10+n01,p=.5).pvalue) if (n10+n01) else 1.0
        pair_p.append((i,j,pval,n10+n01))
raw_p=np.array([r[2] for r in pair_p]); holm=np.empty(len(raw_p))
ix=np.argsort(raw_p)
ordered=np.maximum.accumulate((len(raw_p)-np.arange(len(raw_p)))*raw_p[ix])
holm[ix]=np.minimum(1,ordered)
fig,ax=plt.subplots(figsize=(7.1,5.8))
mat=np.full((len(ORDER),len(ORDER)),np.nan)
for t,(i,j,pval,n_disc) in enumerate(pair_p):mat[i,j]=max(0,-np.log10(max(holm[t],1e-15)))
img=ax.imshow(np.ma.masked_invalid(mat),cmap=LinearSegmentedColormap.from_list('holm',['#f5f6f6',TEAL]),vmin=0,vmax=3)
for t,(i,j,pval,n_disc) in enumerate(pair_p):
    ax.text(j,i,f'p={holm[t]:.3g}\nn={n_disc}',ha='center',va='center',fontsize=7.5,color='white' if mat[i,j]>1.4 else INK)
ax.set_xticks(range(len(ORDER)),ORDER);ax.set_yticks(range(len(ORDER)),ORDER)
ax.grid(False);ax.set_title('FIGURE 29  |  Paired model differences (Holm-corrected)',loc='left',fontweight='bold',pad=12)
fig.colorbar(img,ax=ax,label='-log10(adjusted p)',fraction=.038,pad=.03)
export(fig,'Figure_29_McNemar_Holm_Comparison')

# 30 | Context-only cross-corpus comparison (not a controlled language effect).
# Source: https://doi.org/10.3390/informatics8040079 (published 17 November 2021).
PUBLISHED_2021 = pd.DataFrame({
    'RC':[95.2178,95.1279,95.1022,94.5494,94.5751],
    'DT':[89.3773,89.2530,89.7459,86.4421,87.5176],
    'RF':[94.3350,94.3907,94.3907,92.4869,92.5183],
    'SVC':[95.6206,95.6206,95.6208,94.6337,94.7936],
    'LR':[95.0936,95.0936,95.0938,94.1123,94.2152],
    'NB':[71.4402,71.4402,71.4402,78.3005,78.3005]},
    index=['Grid','Random','Bayesian','PSO','GA']).T
PUBLISHED_2021.to_csv(OUT/'published_2021_reference.csv')
method_list=['Grid','Random','Bayesian','PSO','GA']
repro_matrix=np.full((len(ORDER),len(method_list)),np.nan)
for i,m in enumerate(ORDER):
    for j,meth in enumerate(method_list):
        sub=search_summary[(search_summary.Model==m)&(search_summary.Method==meth)]
        if len(sub) and pd.notna(sub.iloc[0]['CV Accuracy']):
            repro_matrix[i,j]=sub.iloc[0]['CV Accuracy']*100
published_matrix=PUBLISHED_2021.reindex(index=ORDER,columns=method_list).values
new_minus_paper=repro_matrix-published_matrix
fig,ax=plt.subplots(figsize=(8.9,5.1))
scale=15
from matplotlib.colors import TwoSlopeNorm
cmap=LinearSegmentedColormap.from_list('delta',[CORAL,'#f7f8f8',TEAL])
im=ax.imshow(np.ma.masked_invalid(new_minus_paper),cmap=cmap,norm=TwoSlopeNorm(vmin=-scale,vcenter=0,vmax=scale),aspect='auto')
for i in range(len(ORDER)):
    for j in range(len(method_list)):
        v=new_minus_paper[i,j]
        ax.text(j,i,f'{v:+.2f}' if np.isfinite(v) else 'N/A',ha='center',va='center',
            color='white' if np.isfinite(v) and abs(v)>8 else INK,fontsize=9)
ax.set_xticks(range(len(method_list)),method_list)
ax.set_yticks(range(len(ORDER)),ORDER)
ax.set_title('FIGURE 30  |  Russian CV − Arabic 2021 (non-comparable)',loc='left',fontweight='bold',pad=12)
ax.set_xlabel('Search method');ax.set_ylabel('Model');ax.grid(False)
fig.colorbar(im,ax=ax,label='Difference (percentage points)',fraction=.045,pad=.03)
export(fig,'Figure_30_Our_vs_Published_2021')

## 9 · Russian-specific extensions: stop-word ablation and lexical signals

**Figure 31** evaluates the effect of the user-provided **76-word** sentiment-aware stop-word list on Logistic Regression using exactly the same training-only CV folds as the baseline (`stop words` vs `no stop words`). Both variants use the same word bi-gram TF-IDF; neither uses the held-out test for selection. **Figure 32** illustrates the most influential positive and negative **Russian** word/n-gram coefficients learned by a Logistic Regression model fitted to the training split. Coefficients describe model associations, **not causal explanations**. The headings, legends, and figure captions are English.

In [ ]:
# ==========================================================
# FIGURE 31: STOP-WORD ABLATION (TRAINING CV ONLY)
# FIGURE 32: RUSSIAN LEXICAL FEATURES (TRAINING ONLY)
# ==========================================================
from sklearn.metrics import make_scorer
from sklearn.feature_extraction.text import TfidfVectorizer

ablation_results=[]
for label,words in [('No stop-word removal',None),('Sentiment-aware 76-word list',STOP_WORDS)]:
    pipeline=Pipeline([
        ('tfidf',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=2,
           max_features=MAX_FEATURES,dtype=np.float32,sublinear_tf=True,
           token_pattern=r'(?u)\b\w\w+\b',stop_words=words)),
        ('model',LogisticRegression(C=1,max_iter=600,solver='lbfgs',random_state=RANDOM_SEED))
    ])
    time0=time.perf_counter()
    scores=cross_validate(pipeline,X_train_txt,y_train,cv=cv,scoring={'acc':'accuracy','macro_f1':'f1_macro'},n_jobs=1)
    for fold,(acc,f1) in enumerate(zip(scores['test_acc'],scores['test_macro_f1']),1):
        ablation_results.append({'Configuration':label,'Fold':fold,'Accuracy':float(acc),
                                 'Macro-F1':float(f1),'Total run time (s)':time.perf_counter()-time0})
ablation_df=pd.DataFrame(ablation_results)
ablation_df.to_csv(OUT/'russian_stopword_ablation_folds.csv',index=False)
ablation_summary=ablation_df.groupby('Configuration',as_index=False).agg(
    {'Accuracy':['mean','std'],'Macro-F1':['mean','std']})
display(ablation_summary)

fig,ax=plt.subplots(figsize=(7.3,4.5))
configs=['No stop-word removal','Sentiment-aware 76-word list']
pos=np.arange(len(configs));w=.33
for j,(metric,color) in enumerate([('Accuracy',TEAL),('Macro-F1',GOLD)]):
    values=np.array([ablation_df.loc[ablation_df.Configuration==cfg,metric].mean()*100 for cfg in configs])
    stds=np.array([ablation_df.loc[ablation_df.Configuration==cfg,metric].std(ddof=1)*100 for cfg in configs])
    ax.bar(pos + (j-.5)*w,values,w,color=color,label=metric)
    ax.errorbar(pos + (j-.5)*w,values,yerr=stds,fmt='none',ecolor=INK,capsize=3,lw=1)
    for xx,yy in zip(pos+(j-.5)*w,values):ax.text(xx,yy+.7,f'{yy:.1f}',ha='center',fontsize=8)
ax.set_xticks(pos,configs);ax.set_ylim(0,105);ax.legend(frameon=False,ncol=2)
label_axes(ax,'FIGURE 31  |  Russian stop-word ablation (training CV)',ylabel='Mean score ± SD across folds (%)')
export(fig,'Figure_31_Russian_Stopword_Ablation')

# Russian n-grams associated with each class, not translated to preserve feature identity.
lexical_pipeline=build_estimator('LR').fit(X_train_txt,y_train)
features=lexical_pipeline.named_steps['tfidf'].get_feature_names_out()
coef=lexical_pipeline.named_steps['model'].coef_.ravel()
feature_df=pd.DataFrame({'ngram':features,'coefficient':coef})
feature_df.to_csv(OUT/'russian_lexical_coefficients.csv',index=False,encoding='utf-8-sig')
n_show=min(10,len(coef)//2)
neg_idx=np.argsort(coef)[:n_show];pos_idx=np.argsort(coef)[-n_show:][::-1]
idx=np.concatenate([neg_idx,pos_idx])
plot_df=feature_df.iloc[idx].sort_values('coefficient')
fig,ax=plt.subplots(figsize=(8.4,7.0))
ax.barh(plot_df.ngram,plot_df.coefficient,color=[CORAL if c<0 else TEAL for c in plot_df.coefficient])
ax.axvline(0,color=GREY,lw=.8)
label_axes(ax,'FIGURE 32  |  Russian word features associated with sentiment',
          xlabel='Logistic regression coefficient (negative ← 0 → positive)')
ax.grid(axis='x',alpha=.5);ax.grid(axis='y',visible=False)
export(fig,'Figure_32_Russian_Lexical_Signals')


## 10 · Export, data provenance and reproducibility record

Output folders contain CSV files, graph sources and an experiment manifest. These results are specific to the 2023 Russian hotel-review subset and **must not be merged** with the previous Arabic `RSAC_2026_results.zip` or interpreted as directly matching the 2021 article. The last cell also creates a single ZIP of all generated results. Use the optional download cell for one-click export in Colab.

In [ ]:
meta={
    'title':'Russian Hotel Sentiment Analysis: Cross-Language Reproduction and Modern Extension (2026)',
    'reference':'Elgeldawi et al., Informatics 2021 8(4) 79',
    'reference_doi':'10.3390/informatics8040079',
    'source_url':DATASET_SOURCE,
    'source_description':DATASET_METHODOLOGY,
    'source_file':str(source_path),
    'source_sha256':hashlib.sha256(source_path.read_bytes()).hexdigest(),
    'stopwords_file':str(stopwords_path),
    'stopwords_sha256':hashlib.sha256(stopwords_path.read_bytes()).hexdigest(),
    'stopwords_count':int(len(STOP_WORDS)),
    'stopwords_used':bool(USE_STOPWORDS),
    'labels':'rating-derived proxies: negative 1-2 stars -> 0; positive 4-5 stars -> 1',
    'label_annotation':'No manual review-label verification in the supplied text/label CSV.',
    'n_samples_after_cleaning':int(len(labels)),
    'class_counts_after_cleaning':{str(k):int(v) for k,v in labels.value_counts().sort_index().items()},
    'train_size':int(len(y_train)),
    'test_size':int(len(y_test)),
    'profile':PROFILE,'cv_folds':CV_FOLDS,'search_budget':SEARCH_BUDGET,'seed':RANDOM_SEED,
    'scikit_learn_version':__import__('sklearn').__version__,
    'limit':'Hotel ID/rating/organization metadata are unavailable in this uploaded ML CSV; group leakage cannot be ruled out.',
    'comparison':'Our Russian 2023 rating-labeled subset differs from 2021 Arabic manually annotated RSAC.'
}
(OUT/'experiment_manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
print('FINAL:',len(list(FIG_DIR.glob('*.png'))),'figures (each as PNG, SVG, PDF)')
print('Output folder:',OUT.resolve())
import shutil
archive_path=Path('RHSA_2026_results.zip')
shutil.make_archive(str(archive_path.with_suffix('')),'zip',root_dir=OUT)
print('Download archive ready:',archive_path.resolve(),'|',archive_path.stat().st_size,'bytes')


## 11 · Download the results (optional)

After running the notebook, execute the next cell to download **all 28 figures (numbered 05–32; PNG/SVG/PDF), CSV metrics, and the experiment manifest** as one ZIP file. This cell has no effect outside Colab.

In [ ]:
# OPTIONAL: In Google Colab, uncomment to download the entire output archive.
# from google.colab import files
# files.download('RHSA_2026_results.zip')


## 12 · Publish results back to GitHub (optional)

Run the final code cell only after completing the experiment. In Google Colab open **Secrets** (key icon), add `RHSA_GITHUB_TOKEN` with permission to this repository (**Contents: Read and write**), and enable notebook access to it. The token is never written into the repository. Figures and metrics are committed to a distinct timestamped folder under `results/experiments/`. Publishing is **opt-in** and does not overwrite prior runs.

In [ ]:
# Publish the generated results only when explicitly enabled.
PUBLISH_TO_GITHUB = False
if PUBLISH_TO_GITHUB:
    from google.colab import userdata
    import urllib.request
    from pathlib import Path
    script_path = Path('publish_results.py')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Koyd1/RHSA_2026/main/scripts/publish_results.py',
        script_path
    )
    from publish_results import publish_results
    publish_results('RHSA_2026_outputs', token=userdata.get('RHSA_GITHUB_TOKEN'))
else:
    print('GitHub publishing disabled. Set PUBLISH_TO_GITHUB = True to upload results.')
